In [1]:
import pandas as pd
import numpy as np
import pickle
import os
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8')

class DesertCropRecommender:
    def __init__(self, db_path='../models/expert_system_db.pkl'):
        if os.path.exists(db_path):
            with open(db_path, 'rb') as f:
                self.crops_db = pickle.load(f)
            print(f"✅ Loaded {len(self.crops_db)} crops from {db_path}")
        else:
            raise FileNotFoundError(f"❌ Model DB not found at {db_path}. Run 02_feature_engineering.ipynb first!")

    def _score_crop(self, crop, sensor_data):
        score = 0
        total = 0

        # 1. Temperature Match (Weight: 25%)
        temp = sensor_data.get('temp', 30)
        if crop['temp_min'] <= temp <= crop['temp_max']:
            score += 25
        elif abs(temp - crop['temp_min']) <= 5 or abs(temp - crop['temp_max']) <= 5:
            score += 12
        total += 25

        # 2. Humidity Match (Weight: 20%)
        humidity = sensor_data.get('humidity', 50)
        if crop['humidity_min'] <= humidity <= crop['humidity_max']:
            score += 20
        elif abs(humidity - crop['humidity_min']) <= 10 or abs(humidity - crop['humidity_max']) <= 10:
            score += 10
        total += 20

        # 3. Soil Moisture Match (Weight: 20%)
        soil = sensor_data.get('soil_moisture', 40)
        if crop['soil_moisture_min'] <= soil <= crop['soil_moisture_max']:
            score += 20
        elif abs(soil - crop['soil_moisture_min']) <= 10 or abs(soil - crop['soil_moisture_max']) <= 10:
            score += 10
        total += 20

        # 4. pH Match (Weight: 15%)
        ph = sensor_data.get('ph', 6.5)
        if crop['ph_min'] <= ph <= crop['ph_max']:
            score += 15
        elif abs(ph - crop['ph_min']) <= 0.5 or abs(ph - crop['ph_max']) <= 0.5:
            score += 7
        total += 15

        # 5. Light Match (Weight: 10%)
        light = sensor_data.get('light', 70)
        if crop['light_min'] <= light <= crop['light_max']:
            score += 10
        total += 10

        # 6. Rain Match (Weight: 10%)
        rain = sensor_data.get('rain_annual_mm', 500)
        if crop['rain_min'] <= rain <= crop['rain_max']:
            score += 10
        total += 10

        # Priority Bonus (Saudi Arabia / India crops)
        priority_bonus = crop.get('priority', 5) * 0.5
        score += priority_bonus

        # Calculate final percentage
        pct = round((score / (total + 5)) * 100, 1)
        return min(pct, 99.9)

    def predict(self, sensor_data, top_n=3):
        results = {'Tree': [], 'Plant': [], 'Dry Fruit': []}

        for crop in self.crops_db:
            match_pct = self._score_crop(crop, sensor_data)
            
            crop_info = {
                'name': crop['name'],
                'region': crop['region'],
                'match_pct': match_pct,
                'grow_days': crop['grow_days'],
                'water_need': crop['water_need'],
                'category': crop.get('category', 'Plant')
            }
            
            cat = crop_info['category']
            if cat in results:
                results[cat].append(crop_info)

        # Sort each category by highest match score
        for cat in results:
            results[cat].sort(key=lambda x: x['match_pct'], reverse=True)
            results[cat] = results[cat][:top_n]

        return results

# Initialize Model Engine
recommender = DesertCropRecommender()

✅ Loaded 250 crops from ../models/expert_system_db.pkl


In [2]:
# Simulated Test Environments
test_scenarios = {
    'Hot Desert (Riyadh / Rajasthan)': {'temp': 42.0, 'humidity': 18.0, 'soil_moisture': 20.0, 'ph': 7.8, 'light': 92, 'rain_annual_mm': 100},
    'Moderate Humid (Mumbai / Kerala)': {'temp': 28.5, 'humidity': 80.0, 'soil_moisture': 70.0, 'ph': 6.2, 'light': 65, 'rain_annual_mm': 1800},
    'Controlled Hydroponics Greenhouse': {'temp': 24.0, 'humidity': 55.0, 'soil_moisture': 50.0, 'ph': 6.5, 'light': 75, 'rain_annual_mm': 500}
}

print("="*65)
print(" 🔬 EXPERT SYSTEM MODEL STRESS TEST ")
print("="*65)

for scenario_name, env_data in test_scenarios.items():
    print(f"\n🌍 Scenario: {scenario_name}")
    print(f"   Input -> Temp: {env_data['temp']}°C | Hum: {env_data['humidity']}% | Soil: {env_data['soil_moisture']}% | pH: {env_data['ph']}")
    
    predictions = recommender.predict(env_data, top_n=2)
    for cat, crops in predictions.items():
        top_crop = crops[0] if crops else {'name': 'N/A', 'match_pct': 0}
        print(f"   ➜ Top {cat:10s}: {top_crop['name']:25s} | Match: {top_crop['match_pct']}%")
print("="*65)

 🔬 EXPERT SYSTEM MODEL STRESS TEST 

🌍 Scenario: Hot Desert (Riyadh / Rajasthan)
   Input -> Temp: 42.0°C | Hum: 18.0% | Soil: 20.0% | pH: 7.8
   ➜ Top Tree      : Date Palm                 | Match: 99.9%
   ➜ Top Plant     : Pearl Millet (Bajra)      | Match: 71.4%
   ➜ Top Dry Fruit : Ajwa Date                 | Match: 99.9%

🌍 Scenario: Moderate Humid (Mumbai / Kerala)
   Input -> Temp: 28.5°C | Hum: 80.0% | Soil: 70.0% | pH: 6.2
   ➜ Top Tree      : Oud (Agarwood)            | Match: 99.9%
   ➜ Top Plant     : Ginger                    | Match: 99.9%
   ➜ Top Dry Fruit : Cashew (Kaju)             | Match: 90.5%

🌍 Scenario: Controlled Hydroponics Greenhouse
   Input -> Temp: 24.0°C | Hum: 55.0% | Soil: 50.0% | pH: 6.5
   ➜ Top Tree      : Mango                     | Match: 99.9%
   ➜ Top Plant     : Green Bean                | Match: 99.5%
   ➜ Top Dry Fruit : Prune Plum (Aloo Bukhara) | Match: 99.5%


In [3]:
# Save model instance into models directory
model_export_path = '../models/crop_recommender_model.pkl'

with open(model_export_path, 'wb') as f:
    pickle.dump(recommender, f)

print(f"✅ Trained Recommender Model saved at: {model_export_path}")

✅ Trained Recommender Model saved at: ../models/crop_recommender_model.pkl
